In [1]:
%load_ext autoreload
%autoreload 2

from setup_imports import *  # noqa: F401,F403
from langcodes import Language
from phrases.search import get_phrases_by_collection, get_phrase, get_phrase_by_english
from anki_tools import create_anki_deck
from phrases.phrase_model import Phrase

COLLECTION = "LM1000"
DECK = "Pack02"

# Single phrase modification

In [ ]:
p = get_phrase_by_english("It's a new world record")

In [ ]:
p.key

In [ ]:
p = Phrase.create(english_phrase="It's a new world record")

In [ ]:
p.deck = "Pack02"
p.collection = "LM1000"

In [ ]:
p.translate("sv-SE", refine=True)

In [ ]:
p.upload()

In [ ]:
phrases = get_phrases_by_collection(COLLECTION, DECK)

In [ ]:
TARGET_LANGUAGE = Language.get("fr-FR")
print(TARGET_LANGUAGE.display_name())

In [ ]:
p.translations[TARGET_LANGUAGE.to_tag()].text

In [ ]:
p = get_phrase("i_follow_that_channel_cb7ca2")
p.key

In [ ]:
p.translate(
    target_language=TARGET_LANGUAGE,
    overwrite=True,
    refine=False,
    translated_text="Je suis abonnée à cette chaîne",
)
p.generate_audio("flashcard", TARGET_LANGUAGE, overwrite=True, local=True)
p.upload(TARGET_LANGUAGE, overwrite=True)

In [ ]:
p.get_audio(TARGET_LANGUAGE, "flashcard", "normal", local=True)

In [ ]:
# shuffle(phrases)  # otherwise it's alphabetical
target_language_tag = TARGET_LANGUAGE.to_tag()
language_name = TARGET_LANGUAGE.language_name()
create_anki_deck(
    [p],
    source_language="en-GB",
    target_language=TARGET_LANGUAGE,
    output_path=f"../outputs/decks/{target_language_tag}/{COLLECTION}-{DECK}.apkg",
    deck_name=f"FirePhrase - {language_name}::{COLLECTION}::{DECK}",
)

In [ ]:
[str(p) for p in phrases]

# Add vocab and verbs to each translation

In [3]:
# %% --- config -----------------------------------------------------------
from langcodes import Language
from connections.gcloud_auth import get_firestore_client
from nlp import get_verbs_and_vocab


In [8]:

target_language = Language.get("fr-FR")
LANG_TAG   = target_language.to_tag()    # the exact Firestore document key for Swedish translations
LANG_CODE = target_language.language
OVERWRITE = False          # set True to recompute even if verbs already exist
DRY_RUN   = False          # set True to print what would change without writing

db = get_firestore_client("firephrases")

# %% --- helpers ----------------------------------------------------------

def _needs_update(translation_data: dict) -> bool:
    """Return True if verbs or vocab are absent / empty."""
    verbs = translation_data.get("verbs", [])
    vocab = translation_data.get("vocab", [])
    return not verbs and not vocab


# %% --- main loop --------------------------------------------------------

phrase_docs    = list(db.collection("phrases").stream())
total_phrases  = len(phrase_docs)

updated   = 0
skipped   = 0
not_found = 0

print(f"Found {total_phrases} phrases to inspect.\n")

for i, phrase_doc in enumerate(phrase_docs, start=1):
    phrase_hash = phrase_doc.id

    # Direct lookup — we know the key is always "sv-SE"
    t_ref = phrase_doc.reference.collection("translations").document(LANG_TAG)
    t_doc = t_ref.get()

    if not t_doc.exists:
        not_found += 1
        continue

    t_data = t_doc.to_dict()
    text   = t_data.get("text", "")

    if not text:
        print(f"  [{i}/{total_phrases}] {phrase_hash[:12]}… | {LANG_TAG} — ⚠️  no text, skipping")
        skipped += 1
        continue

    if not OVERWRITE and not _needs_update(t_data):
        existing_v = t_data.get("verbs", [])
        existing_w = t_data.get("vocab", [])
        print(
            f"  [{i}/{total_phrases}] {phrase_hash[:12]}… | {LANG_TAG} — ✓ already has "
            f"{len(existing_v)} verbs, {len(existing_w)} vocab — skipping"
        )
        skipped += 1
        continue

    # Run spaCy NLP (sv model loaded/cached on first call)
    result = get_verbs_and_vocab([text], LANG_CODE)
    verbs  = result["verbs"]
    vocab  = result["vocab"]

    print(
        f"  [{i}/{total_phrases}] {phrase_hash[:12]}… | {LANG_TAG} — \"{text[:50]}\"\n"
        f"    verbs={verbs}\n"
        f"    vocab={vocab}"
    )

    if not DRY_RUN:
        t_ref.update({"verbs": verbs, "vocab": vocab})

    updated += 1

# %% --- summary ----------------------------------------------------------
print("\n" + "=" * 60)
print(f"Done.")
print(f"  Phrases inspected : {total_phrases}")
print(f"  Swedish not found : {not_found}")
print(f"  Updated           : {updated}{'  (DRY RUN — nothing written)' if DRY_RUN else ''}")
print(f"  Skipped (ok)      : {skipped}")

[autoreload of src.phrases.search failed: Traceback (most recent call last):
  File "y:\Python Scripts\audio-language-trainer\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "y:\Python Scripts\audio-language-trainer\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 545, in maybe_reload_module
    new_source_code = f.read()
                      ^^^^^^^^
  File "C:\Users\i5\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 5191: character maps to <undefined>
]


Found 2436 phrases to inspect.

(y) Natural Language API client initialized
  [1/2436] a_beer_a63bd… | fr-FR — "Une bière"
    verbs=[]
    vocab=['bière', 'un']
  [2/2436] a_better_opt… | fr-FR — "Une meilleure option"
    verbs=[]
    vocab=['option', 'meilleur', 'un']
  [3/2436] a_bike_in_a_… | fr-FR — "Un vélo en petite vitesse"
    verbs=[]
    vocab=['vitesse', 'un', 'petit', 'vélo', 'en']
  [6/2436] a_bottle_of_… | fr-FR — "Une bouteille de vin"
    verbs=[]
    vocab=['de', 'bouteille', 'vin', 'un']
  [7/2436] a_bottle_und… | fr-FR — "Une bouteille sous le banc"
    verbs=[]
    vocab=['le', 'banc', 'sous', 'un', 'bouteille']
  [8/2436] a_boy_and_a_… | fr-FR — "Un garçon et une fille"
    verbs=[]
    vocab=['fille', 'et', 'garçon', 'un']
  [9/2436] a_bright_col… | fr-FR — "Une couleur vive"
    verbs=[]
    vocab=['couleur', 'vif', 'un']
  [14/2436] a_card_for_e… | fr-FR — "Une carte pour chaque personne"
    verbs=[]
    vocab=['chaque', 'carte', 'un', 'personne', 'pour']
  [

# Add tags

In [9]:
from google.cloud import firestore
from connections.gcloud_auth import get_firestore_client

def migrate_tags(database_name: str = "firephrases", dry_run: bool = True):
    db = get_firestore_client(database_name)
    phrases_ref = db.collection("phrases")
    
    print(f"Starting tag migration (Dry Run: {dry_run})...")
    
    count = 0
    updated_translations = 0
    
    # Stream all phrases
    for phrase_doc in phrases_ref.stream():
        phrase_data = phrase_doc.to_dict()
        
        # Get collection and deck, format them safely as tags
        collection = phrase_data.get("collection")
        deck = phrase_data.get("deck")
        
        tags_to_add = []
        if collection:
            tags_to_add.append(collection.replace(" ", "_"))
        if deck:
            tags_to_add.append(deck.replace(" ", "_"))
            
        if not tags_to_add:
            continue
            
        count += 1
        
        # Stream all translation documents for this phrase
        for t_doc in phrase_doc.reference.collection("translations").stream():
            if not dry_run:
                # Use ArrayUnion to safely append without overwriting existing
                # tags or creating duplicates
                t_doc.reference.update({
                    "tags": firestore.ArrayUnion(tags_to_add)
                })
            updated_translations += 1
            
        if count % 100 == 0:
            print(f"Processed {count} phrases...")
            
    print(f"Done! Processed {count} phrases, updating {updated_translations} translation docs.")

[autoreload of src.phrases.search failed: Traceback (most recent call last):
  File "y:\Python Scripts\audio-language-trainer\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "y:\Python Scripts\audio-language-trainer\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 545, in maybe_reload_module
    new_source_code = f.read()
                      ^^^^^^^^
  File "C:\Users\i5\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 5191: character maps to <undefined>
]


In [10]:
migrate_tags(dry_run=False)

Starting tag migration (Dry Run: False)...
Processed 100 phrases...
Processed 200 phrases...
Processed 300 phrases...
Processed 400 phrases...
Processed 500 phrases...
Processed 600 phrases...
Processed 700 phrases...
Processed 800 phrases...
Processed 900 phrases...
Processed 1000 phrases...
Processed 1100 phrases...
Processed 1200 phrases...
Processed 1300 phrases...
Processed 1400 phrases...
Processed 1500 phrases...
Processed 1600 phrases...
Processed 1700 phrases...
Done! Processed 1763 phrases, updating 6134 translation docs.
